In [ ]:
from __future__ import annotations
import torch
import torch.nn as nn
from datasets import Dataset, load_dataset
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import Blip2ForConditionalGeneration, Blip2Processor

In [2]:
SEED = 42
MODEL_ID = "Salesforce/blip2-opt-2.7b"
DATASET_ID = "reach-vb/pokemon-blip-captions"

TRAIN_SAMPLE_NUM = 128
TEST_SAMPLE_NUM = 16

BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
MAX_TRAIN_BATCHES = 20 #for short training, just for debugging purpose

LEARNING_RATE = 1e-5
MAX_TEXT_LENGTH = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.cuda.manual_seed_all(SEED)

In [3]:
processor = Blip2Processor.from_pretrained(MODEL_ID)
model = Blip2ForConditionalGeneration.from_pretrained(MODEL_ID).to(device)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

In [4]:
raw_dataset:Dataset = load_dataset(DATASET_ID, split="train")
data_split = raw_dataset.train_test_split(test_size=0.1, seed=SEED)

train_data = data_split["train"].shuffle(seed=SEED)
test_data = data_split["test"]

print(len(train_data))
print(len(test_data))

749
84


In [5]:
train_data_chunk = train_data.select(range(TRAIN_SAMPLE_NUM))
test_data_chunk = test_data.select(range(TEST_SAMPLE_NUM))

In [6]:
from typing import Any
from torch import Tensor

class Blip2CaptionCollator:
    def __init__(self, processor:Blip2Processor, model:Blip2ForConditionalGeneration, max_text_length:int):
        self.processor = processor
        self.model = model
        self.max_text_length = max_text_length
    
    def __call__(self, cur_data: list[dict]) -> dict[str, Tensor]:
        images = []
        captions = []
        
        for elem in cur_data:
            images.append(elem["image"].convert("RGB"))
            captions.append(elem["text"].strip())
        
        batch = self.processor(images, captions, padding=True, truncation=True, 
                               max_length=self.max_text_length, return_tensors="pt")
        
        labels = batch["input_ids"].clone() #hugging face will deal shift right
        labels[batch["attention_mask"] == 0] = -100 #pad token for hf model
        
        #if there's image place holder, we should not calculate llm loss for image token
        image_token_index = getattr(self.model.config, "image_token_index", None)
        if image_token_index is not None:
            labels[batch["input_ids"] == image_token_index] = -100
        
        batch["labels"] = labels
        
        return batch
    
collator = Blip2CaptionCollator(processor, model, MAX_TEXT_LENGTH)
train_loader = DataLoader(train_data_chunk, BATCH_SIZE, shuffle=True, collate_fn=collator, pin_memory=True)
test_loader = DataLoader(test_data_chunk, BATCH_SIZE, collate_fn=collator, pin_memory=True)


In [7]:
def move_batch_to_device(batch:dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
    moved_batch = {key:value.to(device) for key, value in batch.items()}
    
    moved_batch["pixel_values"] = moved_batch["pixel_values"].to(dtype=torch.bfloat16)
    
    return moved_batch

def autocast_context():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16)

In [8]:
#it's for debug purpose, ai generated
def inspect_blip2_shapes() -> None:
    model.eval()

    example = test_data_chunk[0]
    image = example["image"].convert("RGB")

    inputs = processor(
        images=image,
        return_tensors="pt",
    )

    pixel_values = inputs["pixel_values"].to(
        device=device,
        dtype=torch.bfloat16,
    )

    with torch.inference_mode():
        with autocast_context():
            image_features = model.get_image_features(
                pixel_values=pixel_values
            )

    vision_output = (
        image_features.vision_outputs.last_hidden_state
    )

    qformer_output = (
        image_features.qformer_outputs.last_hidden_state
    )

    llm_prefix = image_features.pooler_output

    print("\nTensor shapes")
    print(
        "pixel_values:",
        tuple(pixel_values.shape),
    )

    print(
        "Vision encoder output:",
        tuple(vision_output.shape),
    )

    print(
        "Q-Former output:",
        tuple(qformer_output.shape),
    )

    print(
        "Projected LLM prefix:",
        tuple(llm_prefix.shape),
    )


inspect_blip2_shapes()


Tensor shapes
pixel_values: (1, 3, 224, 224)
Vision encoder output: (1, 257, 1408)
Q-Former output: (1, 32, 768)
Projected LLM prefix: (1, 32, 2560)


In [ ]:
from PIL import Image

@torch.inference_mode()
def generate_caption(image: Image.Image) -> str:
    model.eval() #for disable dropout and normalization

    inputs = processor(images=image.convert("RGB"), return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    inputs["pixel_values"] = inputs["pixel_values"].to(dtype=torch.bfloat16)
    
    with autocast_context():
        generated_ids = model.generate(**inputs, max_new_tokens=30, num_beams=3, repetition_penalty=1.1)
    
    #it returns single element list, so in torder to access str, i need [0]
    caption = processor.batch_decode(generated_ids, skip_special_tokens = True)[0] 
    
    return caption.strip()


In [10]:
print("before finetune")
for index in range(min(3, len(test_data_chunk))):
    i = test_data_chunk[index]
    pred = generate_caption(i["image"])
    
    print(f"\nExample {index}")
    print("Target:   ", i["text"])
    print("Generated:", pred)

before finetune

Example 0
Target:    a yellow and white cartoon character with a red eye
Generated: a yellow and white pokemon is standing on its hind legs

Example 1
Target:    a black and white bird with a red beak
Generated: a cartoon bird with black and white feathers

Example 2
Target:    a drawing of a pokemon with a green leaf on it's back
Generated: a pokemon with large green leaves on its back


In [11]:
for parameter in model.parameters():
    parameter.requires_grad = False

#train q former
for parameter in model.qformer.parameters():
    parameter.requires_grad = True

#train query token
model.query_tokens.requires_grad = True

#q former to llm
for parameter in model.language_projection.parameters():
    parameter.requires_grad = True

optimizer = AdamW(parameter for parameter in model.parameters() if parameter.requires_grad)

#no need for kv cache during training
model.language_model.config.use_cache = False

In [12]:
model.train()

#freeze vision model and language model
model.vision_model.eval()
model.language_model.eval()

model.qformer.train()
model.language_projection.train()

optimizer.zero_grad(set_to_none=True)

for batch_index, batch in enumerate(train_loader):
    if batch_index >= MAX_TRAIN_BATCHES:
        break
    
    batch = move_batch_to_device(batch)
    
    with autocast_context():
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    
    print(f"i:{batch_index}, loss:{loss}")
    

i:0, loss:1.2786957025527954
i:1, loss:4.235738277435303
i:2, loss:8.55555534362793
i:3, loss:7.017361164093018
i:4, loss:7.9289774894714355
i:5, loss:7.18359375
i:6, loss:7.080468654632568
i:7, loss:6.444531440734863
i:8, loss:5.559244632720947
i:9, loss:5.768012046813965
i:10, loss:5.5380859375
i:11, loss:5.8971943855285645
i:12, loss:5.623046875
i:13, loss:9.419921875
i:14, loss:5.341145992279053
i:15, loss:6.42274284362793
i:16, loss:5.549278736114502
i:17, loss:5.40625
i:18, loss:11.537500381469727
i:19, loss:5.493750095367432


In [ ]:
save_path = Path("blip2_qformer_adapter.pt")

trainable_state_dict = {
    name: parameter.detach().cpu()
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
}

torch.save(
    trainable_state_dict,
    save_path,
)

print(f"\nSaved trainable weights to: {save_path}")